# 📊 01_Quater – Travailler avec des fichiers Excel

---

## 🎯 Objectifs
- Comprendre la différence entre un fichier **CSV** et un fichier **Excel**
- Lire un fichier `.xlsx` avec `pd.read_excel()`
- Gérer les problèmes courants : **feuilles multiples**, **lignes de titre parasites**, **types mal détectés**
- Écrire un fichier Excel propre avec `to_excel()`
- Écrire dans **plusieurs feuilles** d'un même classeur

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import pandas as pd
print("Pandas version :", pd.__version__)

# Pour lire des fichiers Excel, Pandas a besoin de openpyxl
# Si vous avez une erreur, installez-le avec : pip install openpyxl
try:
    import openpyxl
    print("openpyxl version :", openpyxl.__version__, "✅")
except ImportError:
    print("❌ openpyxl manquant — installez-le : pip install openpyxl")

---
## 📝 Partie 1 – CSV vs Excel : quelle différence ?

### 🔎 Deux formats très différents

| | CSV | Excel (`.xlsx`) |
|-|-----|----------------|
| **Type de fichier** | Texte brut | Fichier binaire compressé |
| **Feuilles** | Une seule | Plusieurs feuilles possibles |
| **Mise en forme** | Aucune | Couleurs, polices, formules, graphiques |
| **Taille** | Léger | Plus lourd |
| **Compatibilité** | Universel | Principalement Microsoft / Google |
| **Problèmes courants** | Encodage, séparateur | Lignes de titre, types, feuilles multiples |

**Dans la vraie vie :** vos collègues vous envoient souvent des `.xlsx` avec :
- Des **titres sur les 2-3 premières lignes** avant les vraies données
- Des **colonnes fusionnées** dans les en-têtes
- Des **feuilles multiples** (une par mois, une par région…)
- Des **dates mal formatées** ou des **nombres stockés en texte**

Ce notebook vous apprend à gérer tous ces cas.

### 🔎 Lire un fichier Excel

```python
# Lire la première feuille (par défaut)
df = pd.read_excel("fichier.xlsx")

# Lire une feuille spécifique par son nom
df = pd.read_excel("fichier.xlsx", sheet_name="Ventes")

# Lire une feuille spécifique par son numéro (0 = première)
df = pd.read_excel("fichier.xlsx", sheet_name=0)

# Lire TOUTES les feuilles → retourne un dictionnaire
toutes = pd.read_excel("fichier.xlsx", sheet_name=None)
# toutes["Ventes"] → DataFrame de la feuille Ventes
# toutes["Stocks"] → DataFrame de la feuille Stocks
```

In [ ]:
import pandas as pd

# On crée d'abord un fichier Excel de démonstration avec plusieurs feuilles
# (dans un vrai projet ce fichier viendrait de vos collègues)

ventes = pd.DataFrame({
    "Date"    : ["2024-01-15", "2024-01-22", "2024-02-03",
                 "2024-02-18", "2024-03-05", "2024-03-20"],
    "Région"  : ["Nord", "Sud", "Nord", "Est", "Sud", "Nord"],
    "Produit" : ["Pizza", "Burger", "Pizza", "Sushi", "Pizza", "Burger"],
    "Quantité": [120, 85, 140, 60, 175, 95],
    "CA (€)"  : [1500, 765, 1750, 900, 2187, 855]
})

stocks = pd.DataFrame({
    "Produit"       : ["Pizza", "Burger", "Sushi", "Tacos"],
    "Stock_actuel"  : [250, 180, 90, 120],
    "Stock_minimum" : [50, 40, 20, 30],
    "Prix_unitaire" : [12.5, 9.0, 15.0, 9.5]
})

# Écrire dans un fichier Excel avec plusieurs feuilles
with pd.ExcelWriter("rapport_restaurant.xlsx", engine="openpyxl") as writer:
    ventes.to_excel(writer, sheet_name="Ventes",  index=False)
    stocks.to_excel(writer, sheet_name="Stocks",  index=False)

print("✅ Fichier 'rapport_restaurant.xlsx' créé avec 2 feuilles : Ventes et Stocks")

---
## 📝 Partie 2 – Lire et explorer un fichier Excel

### 🔎 Connaître les feuilles disponibles

La première chose à faire quand on reçoit un Excel inconnu : voir combien de feuilles il contient.

```python
# Lister les feuilles sans tout charger
xl = pd.ExcelFile("fichier.xlsx")
print(xl.sheet_names)   # → ['Ventes', 'Stocks', 'Résumé']
```

In [ ]:
import pandas as pd

# Inspecter les feuilles disponibles sans charger tout le fichier
xl = pd.ExcelFile("rapport_restaurant.xlsx")
print("Feuilles disponibles :", xl.sheet_names)

print()

# Lire la feuille Ventes
df_ventes = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Ventes")
print("=== Feuille Ventes ===")
print(df_ventes)

print()

# Lire la feuille Stocks
df_stocks = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Stocks")
print("=== Feuille Stocks ===")
print(df_stocks)

In [ ]:
import pandas as pd

# Lire TOUTES les feuilles en une fois → dictionnaire de DataFrames
toutes_feuilles = pd.read_excel("rapport_restaurant.xlsx", sheet_name=None)

print("Type retourné :", type(toutes_feuilles))
print("Clés (noms des feuilles) :", list(toutes_feuilles.keys()))

print()

# Accéder à chaque feuille par son nom
for nom_feuille, df in toutes_feuilles.items():
    print(f"--- Feuille '{nom_feuille}' : {df.shape[0]} lignes × {df.shape[1]} colonnes ---")
    print(df.head(2))
    print()

---
## 📝 Partie 3 – Problèmes courants avec les fichiers Excel

### 🔎 Problème 1 : lignes de titre avant les données

Très souvent, les fichiers Excel envoyés par des collègues ont un titre sur les premières lignes :

```
Ligne 1 : RAPPORT MENSUEL - Janvier 2024          ← titre parasite
Ligne 2 : Généré le 15/02/2024                    ← sous-titre parasite
Ligne 3 : (vide)
Ligne 4 : Date | Région | Produit | Quantité      ← vraies en-têtes
Ligne 5 : données...
```

→ Pandas lira le titre comme première ligne de données. Solution : `skiprows`.

```python
# Ignorer les 3 premières lignes (0, 1, 2)
df = pd.read_excel("fichier.xlsx", skiprows=3)

# Ou ignorer des lignes spécifiques
df = pd.read_excel("fichier.xlsx", skiprows=[0, 1, 2])
```

### 🔎 Problème 2 : types mal détectés

Pandas devine les types automatiquement, mais il se trompe parfois :
- Une colonne de codes postaux (`75001`) interprétée comme entier
- Une date interprétée comme texte
- Des montants avec `"€"` ou `","` interprétés comme texte

```python
# Forcer le type d'une colonne à la lecture
df = pd.read_excel("fichier.xlsx", dtype={"Code_postal": str, "Téléphone": str})

# Convertir une colonne après lecture
df["Date"] = pd.to_datetime(df["Date"])   # convertir en vrai type date
df["CA"]   = pd.to_numeric(df["CA"], errors="coerce")  # convertir en nombre
```

### 🔎 Problème 3 : sélectionner des colonnes précises

```python
# Ne charger que certaines colonnes (par nom)
df = pd.read_excel("fichier.xlsx", usecols=["Date", "Région", "CA (€)"])

# Ne charger que les N premières lignes
df = pd.read_excel("fichier.xlsx", nrows=100)
```

In [ ]:
import pandas as pd

# Simuler un fichier Excel avec un titre parasite sur 2 lignes
import openpyxl
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Données"

# Lignes parasites (comme un vrai fichier d'entreprise)
ws.append(["RAPPORT MENSUEL - Restaurant La Bella"])
ws.append(["Exporté le 01/04/2024"])
ws.append([])  # ligne vide

# Vraies données
ws.append(["Région", "Produit", "Quantité", "CA"])
ws.append(["Nord", "Pizza", 120, 1500])
ws.append(["Sud",  "Burger", 85,  765])
ws.append(["Est",  "Sushi",  60,  900])
wb.save("rapport_avec_titre.xlsx")

print("=== Lecture SANS skiprows — titre lu comme données ===")
df_mauvais = pd.read_excel("rapport_avec_titre.xlsx")
print(df_mauvais)

print()

print("=== Lecture AVEC skiprows=3 — on saute les 3 premières lignes ===")
df_correct = pd.read_excel("rapport_avec_titre.xlsx", skiprows=3)
print(df_correct)

In [ ]:
import pandas as pd

# Lire le fichier ventes avec gestion des types
df = pd.read_excel(
    "rapport_restaurant.xlsx",
    sheet_name="Ventes",
    usecols=["Date", "Région", "Produit", "CA (€)"]   # seulement 4 colonnes
)

print("=== Avant conversion des types ===")
print(df.dtypes)
print(df.head(3))

print()

# Convertir la colonne Date en vrai type datetime
df["Date"] = pd.to_datetime(df["Date"])

print("=== Après conversion de la date ===")
print(df.dtypes)
print(df.head(3))

print()

# Extraire le mois depuis la date
df["Mois"] = df["Date"].dt.month
df["Mois_nom"] = df["Date"].dt.month_name(locale="fr_FR.utf8")
print("=== Avec colonnes Mois extraites ===")
print(df)

---
## 📝 Partie 4 – Écrire dans un fichier Excel

### 🔎 Exporter un DataFrame en Excel

```python
# Simple — une seule feuille
df.to_excel("resultat.xlsx", index=False)

# Avec nom de feuille personnalisé
df.to_excel("resultat.xlsx", sheet_name="Analyse", index=False)
```

### 🔎 Écrire dans plusieurs feuilles

Pour écrire **plusieurs DataFrames** dans le même fichier Excel (une feuille par DataFrame), on utilise `ExcelWriter` :

```python
with pd.ExcelWriter("rapport.xlsx", engine="openpyxl") as writer:
    df_ventes.to_excel(writer, sheet_name="Ventes",  index=False)
    df_stocks.to_excel(writer, sheet_name="Stocks",  index=False)
    df_resume.to_excel(writer, sheet_name="Résumé",  index=False)
# → 3 feuilles dans le même fichier Excel
```

> 💡 **Le `with` est important :** il garantit que le fichier est bien fermé et sauvegardé après l'écriture — même si une erreur se produit.

In [ ]:
import pandas as pd

df_ventes = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Ventes")
df_stocks = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Stocks")

# Créer un résumé des ventes par région
df_resume = df_ventes.groupby("Région")[["Quantité", "CA (€)"]].sum().reset_index()
df_resume.columns = ["Région", "Total_Quantité", "Total_CA (€)"]

print("=== Résumé par région ===")
print(df_resume)

# Écrire les 3 DataFrames dans un seul fichier Excel
with pd.ExcelWriter("rapport_final.xlsx", engine="openpyxl") as writer:
    df_ventes.to_excel(writer, sheet_name="Ventes détail", index=False)
    df_stocks.to_excel(writer, sheet_name="Stocks",        index=False)
    df_resume.to_excel(writer, sheet_name="Résumé",        index=False)

print()
print("✅ Fichier 'rapport_final.xlsx' créé avec 3 feuilles")

# Vérification : relire et lister les feuilles
xl = pd.ExcelFile("rapport_final.xlsx")
print("Feuilles créées :", xl.sheet_names)

---
## 🧩 Exercice 1 – Lire et explorer un Excel multi-feuilles

Repartez du fichier `rapport_restaurant.xlsx` :

1. Listez les feuilles disponibles sans charger tout le fichier
2. Lisez la feuille `"Ventes"` et affichez ses dimensions et ses types avec `info()`
3. Lisez uniquement les colonnes `"Région"`, `"Produit"` et `"CA (€)"` de la feuille Ventes
4. Calculez le **CA total par Région** avec `groupby()`
5. Lisez la feuille `"Stocks"` et identifiez les produits dont le stock est **en dessous du minimum**
6. Lisez **toutes les feuilles** d'un coup et affichez la shape de chacune

In [ ]:
import pandas as pd

# TODO 1 : lister les feuilles
xl = pd.ExcelFile("rapport_restaurant.xlsx")
print("1. Feuilles disponibles :", ...)

# TODO 2 : lire la feuille Ventes + info()
df_ventes = pd.read_excel("rapport_restaurant.xlsx", sheet_name=...)
print("\n2. Structure de la feuille Ventes :")
df_ventes.info()

# TODO 3 : lire uniquement 3 colonnes
df_partiel = pd.read_excel(
    "rapport_restaurant.xlsx",
    sheet_name="Ventes",
    usecols=[...]
)
print("\n3. Colonnes sélectionnées :")
print(df_partiel)

# TODO 4 : CA total par région
print("\n4. CA total par région :")
print(df_ventes.groupby(...)[...].sum())

# TODO 5 : stocks en dessous du minimum
df_stocks = pd.read_excel("rapport_restaurant.xlsx", sheet_name=...)
print("\n5. Produits sous le stock minimum :")
print(df_stocks[df_stocks[...] < df_stocks[...]])

# TODO 6 : toutes les feuilles
toutes = pd.read_excel("rapport_restaurant.xlsx", sheet_name=...)
print("\n6. Shape de chaque feuille :")
for nom, df in toutes.items():
    print(f"  '{nom}' : {df.shape}")

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

# 1
xl = pd.ExcelFile("rapport_restaurant.xlsx")
print("Feuilles :", xl.sheet_names)   # ['Ventes', 'Stocks']

# 2
df_ventes = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Ventes")
df_ventes.info()
# 6 lignes, 5 colonnes

# 3 — usecols filtre les colonnes dès la lecture
df_partiel = pd.read_excel(
    "rapport_restaurant.xlsx",
    sheet_name="Ventes",
    usecols=["Région", "Produit", "CA (€)"]
)
print(df_partiel)

# 4
print(df_ventes.groupby("Région")["CA (€)"].sum())
# Est    900  Nord  4105  Sud  2952

# 5
df_stocks = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Stocks")
print(df_stocks[df_stocks["Stock_actuel"] < df_stocks["Stock_minimum"]])
# → aucun produit en rupture ici (tous au-dessus du minimum)

# 6 — sheet_name=None charge toutes les feuilles
toutes = pd.read_excel("rapport_restaurant.xlsx", sheet_name=None)
for nom, df in toutes.items():
    print(f"  '{nom}' : {df.shape}")
# 'Ventes' : (6, 5)
# 'Stocks' : (4, 4)
```
</details>

---
## 🧩 Exercice 2 – Analyse complète et export multi-feuilles

En repartant du fichier `rapport_restaurant.xlsx` :

1. Convertissez la colonne `"Date"` en type datetime
2. Ajoutez une colonne `"Mois"` (numéro du mois extrait de la date)
3. Calculez le **CA total par mois** avec `groupby()`
4. Calculez le **CA total par Produit**
5. Identifiez le **produit le plus vendu** (par quantité totale)
6. Créez un fichier `"analyse_finale.xlsx"` avec **3 feuilles** :
   - `"Données"` : le DataFrame ventes complet avec la colonne Mois
   - `"Par_mois"` : le CA total par mois
   - `"Par_produit"` : le CA total par produit

In [ ]:
import pandas as pd

df = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Ventes")

# TODO 1 : convertir Date en datetime
df["Date"] = pd.to_datetime(df[...])

# TODO 2 : ajouter colonne Mois
df["Mois"] = df["Date"]....
print("Aperçu avec Mois :")
print(df[["Date", "Mois", "Produit", "CA (€)"]])

# TODO 3 : CA total par mois
ca_par_mois = df.groupby(...)[[...]].sum().reset_index()
print("\nCA par mois :")
print(ca_par_mois)

# TODO 4 : CA total par produit
ca_par_produit = df.groupby(...)[...].sum().reset_index()
print("\nCA par produit :")
print(ca_par_produit)

# TODO 5 : produit le plus vendu (quantité)
qte_par_produit = df.groupby("Produit")["Quantité"].sum()
print(f"\nProduit le plus vendu : {qte_par_produit.idxmax()} ({qte_par_produit.max()} unités)")

# TODO 6 : export 3 feuilles
with pd.ExcelWriter("analyse_finale.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer,            sheet_name=..., index=False)
    ca_par_mois.to_excel(writer,   sheet_name=..., index=False)
    ca_par_produit.to_excel(writer, sheet_name=..., index=False)

print("\n✅ Fichier 'analyse_finale.xlsx' créé avec 3 feuilles")
print("Feuilles :", pd.ExcelFile("analyse_finale.xlsx").sheet_names)

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

df = pd.read_excel("rapport_restaurant.xlsx", sheet_name="Ventes")

# 1
df["Date"] = pd.to_datetime(df["Date"])

# 2 — .dt.month extrait le numéro du mois depuis un datetime
df["Mois"] = df["Date"].dt.month
print(df[["Date", "Mois", "Produit", "CA (€)"]])

# 3
ca_par_mois = df.groupby("Mois")[["CA (€)"]].sum().reset_index()
print(ca_par_mois)
# Mois 1 : 2265 €   Mois 2 : 2650 €   Mois 3 : 3042 €
# → Le CA augmente chaque mois

# 4
ca_par_produit = df.groupby("Produit")["CA (€)"].sum().reset_index()
print(ca_par_produit)
# Pizza : 5437 €  Burger : 855 €  Sushi : 900 €

# 5
qte_par_produit = df.groupby("Produit")["Quantité"].sum()
print(f"Produit le plus vendu : {qte_par_produit.idxmax()} ({qte_par_produit.max()} unités)")
# → Pizza (435 unités)

# 6 — ExcelWriter avec 3 feuilles
with pd.ExcelWriter("analyse_finale.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer,             sheet_name="Données",    index=False)
    ca_par_mois.to_excel(writer,    sheet_name="Par_mois",   index=False)
    ca_par_produit.to_excel(writer, sheet_name="Par_produit", index=False)

print("Feuilles :", pd.ExcelFile("analyse_finale.xlsx").sheet_names)
# → ['Données', 'Par_mois', 'Par_produit']
```

> 💡 **`.dt.month`** : quand une colonne est de type `datetime`, Pandas expose plein d'accesseurs via `.dt` : `.dt.month` (mois), `.dt.year` (année), `.dt.day` (jour), `.dt.weekday` (jour de la semaine 0=lundi), `.dt.quarter` (trimestre)…
</details>

---
## ✅ Récapitulatif

| Concept | Code | Ce qu'il faut retenir |
|---------|------|-----------------------|
| **Lire un Excel** | `pd.read_excel("f.xlsx")` | Lit la première feuille par défaut |
| **Choisir une feuille** | `sheet_name="Ventes"` | Par nom ou par numéro (0 = première) |
| **Toutes les feuilles** | `sheet_name=None` | Retourne un dictionnaire `{nom: DataFrame}` |
| **Lister les feuilles** | `pd.ExcelFile("f.xlsx").sheet_names` | Sans charger tout le fichier |
| **Sauter des lignes** | `skiprows=3` | Ignorer les titres parasites |
| **Colonnes précises** | `usecols=["col1", "col2"]` | Charger seulement certaines colonnes |
| **Convertir une date** | `pd.to_datetime(df["Date"])` | Transforme un texte en vrai type datetime |
| **Extraire le mois** | `df["Date"].dt.month` | Accesseur `.dt` sur un datetime |
| **Écrire un Excel** | `df.to_excel("f.xlsx", index=False)` | `index=False` évite la colonne parasite |
| **Multi-feuilles** | `pd.ExcelWriter(...)` avec `with` | Écrire plusieurs DataFrames dans un seul fichier |

**Ce que vous savez faire maintenant :**
- Ouvrir n'importe quel fichier Excel reçu de collègues
- Naviguer entre les feuilles
- Corriger les problèmes courants (titres parasites, types, dates)
- Produire un rapport Excel multi-feuilles depuis Python

---
📘 **Prochain notebook → `05` : Visualisation avec Pandas**